# oz-tracker — Opportunity Zone Investment Tracker
## Demo: OZ 1.0 vs OZ 2.0 Analysis & QOF Tax Benefit Modeling

This notebook demonstrates how to use oz-tracker to:
- Check OZ 1.0 and OZ 2.0 eligibility for census tracts
- Compare tract overlap between OZ 1.0 and OZ 2.0 designations
- Calculate QOF tax benefits including rural QORF enhanced incentives
- Compare investment scenarios across all program versions
- Track a portfolio of Qualified Opportunity Fund investments

**Key context:** The One Big Beautiful Bill Act (July 2025) made Opportunity
Zones permanent. OZ 2.0 takes effect January 1, 2027 with stricter eligibility,
enhanced rural benefits, and new reporting requirements.


In [ ]:
import sys
sys.path.insert(0, '..')

from oztracker import (
    OZ1Checker, OZ2Checker,
    OZInvestment, calculate_benefits, compare_scenarios,
    OZPortfolio, FUND_TYPES, OZ_VERSIONS
)
from oztracker.data.loader import check_oz1_eligibility, check_oz2_eligibility
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("oz-tracker loaded successfully")
print(f"\nOZ Program Versions:")
for k, v in OZ_VERSIONS.items():
    print(f"  {k}: {v}")


## 1. OZ 1.0 Eligibility (Current Map — Through 2028)

8,764 census tracts designated under IRS Notice 2018-48.
Current map remains in effect through December 31, 2028.


In [ ]:
oz1 = OZ1Checker()

# Check specific tracts
tracts = [
    ("17031840100", "Chicago South Side"),
    ("17031839100", "Chicago West Side"),
    ("17031010100", "Chicago North Shore"),
    ("26163518300", "Detroit"),
    ("36061015900", "NYC Bronx"),
    ("36047052200", "NYC Brooklyn Affluent"),
    ("13121010400", "Atlanta"),
    ("17019000100", "Rural Illinois"),
]

print(f"\n{'Tract ID':<15} {'Location':<25} {'OZ 1.0 Designated'}")
print("-" * 55)
for tract_id, location in tracts:
    designated = oz1.is_designated(tract_id)
    status = "YES" if designated else "NO"
    print(f"{tract_id:<15} {location:<25} {status}")

print(f"\nTotal OZ 1.0 tracts: {oz1.tract_count:,}")


## 2. OZ 2.0 Eligibility (2027 Designations)

OZ 2.0 uses stricter criteria:
- MFI < **70%** of AMI (vs 80% in OZ 1.0), OR
- Poverty rate >= 20% AND MFI <= 125% of AMI

Contiguous tract rule eliminated — fewer eligible tracts overall.


In [ ]:
oz2 = OZ2Checker()

print(f"\n{'Tract ID':<15} {'Location':<25} {'OZ 2.0 Eligible':<18} {'Rural'}")
print("-" * 65)
for tract_id, location in tracts:
    eligible = oz2.is_eligible(tract_id)
    rural = oz2.is_rural(tract_id)
    print(f"{tract_id:<15} {location:<25} {'YES' if eligible else 'NO':<18} {'YES' if rural else 'NO'}")

print(f"\nTotal OZ 2.0 eligible tracts: {oz2.eligible_tract_count:,}")
print(f"Rural tracts: {oz2.rural_tract_count:,}")


## 3. OZ 1.0 vs OZ 2.0 Tract Comparison

Which tracts are losing OZ status? Which are gaining?
This is critical intelligence for investors evaluating whether
to deploy capital under OZ 1.0 or wait for OZ 2.0.


In [ ]:
comparison = oz2.compare_oz1_oz2(oz1.designated_tracts)


## 4. Eligibility Rule Comparison

Side-by-side comparison of OZ 1.0 vs OZ 2.0 eligibility criteria.


In [ ]:
test_cases = [
    (0.38, 0.55, "High poverty, low AMI"),
    (0.42, 0.48, "Very high poverty, very low AMI"),
    (0.18, 0.92, "Low poverty, high AMI"),
    (0.25, 0.75, "Moderate poverty, moderate AMI"),
    (0.15, 0.68, "Low poverty, below 70% AMI"),
    (0.22, 1.10, "Moderate poverty, above AMI threshold"),
]

print(f"{'Poverty':<10} {'AMI Ratio':<12} {'Description':<35} {'OZ 1.0':<10} {'OZ 2.0'}")
print("-" * 80)
for pr, ami, desc in test_cases:
    oz1_elig = check_oz1_eligibility(pr, ami)
    oz2_elig = check_oz2_eligibility(pr, ami)
    print(f"{pr*100:.0f}%{'':<7} {ami*100:.0f}%{'':<9} {desc:<35} "
          f"{'YES' if oz1_elig else 'NO':<10} {'YES' if oz2_elig else 'NO'}")


## 5. QOF Tax Benefit Calculator — OZ 2.0 Standard

A $500k capital gain invested in an OZ 2.0 QOF on March 15, 2027.
Held for 10 years and exited on March 15, 2037.


In [ ]:
standard_inv = OZInvestment(
    id="INV001",
    investor_name="Jay Patel",
    fund_name="Midwest OZ Fund I",
    fund_type="qof",
    oz_version="oz2",
    tract_id="17031840100",
    investment_type="real_estate",
    capital_gain_invested=500_000,
    investment_date="2027-03-15",
    fmv_at_investment=500_000,
    current_fmv=750_000,
    state="IL",
    is_rural=False,
)

benefits_std = calculate_benefits(
    standard_inv,
    current_fmv=750_000,
    exit_date="2037-03-15",
)
benefits_std.summary()


## 6. Rural QORF — Enhanced 30% Step-Up Benefit

The same investment in a rural Qualified Opportunity Rural Fund (QORF)
receives triple the basis step-up: 30% vs 10%.


In [ ]:
rural_inv = OZInvestment(
    id="INV002",
    investor_name="Jay Patel",
    fund_name="Rural Illinois QORF",
    fund_type="qorf",
    oz_version="oz2",
    tract_id="17019000100",
    investment_type="real_estate",
    capital_gain_invested=500_000,
    investment_date="2027-03-15",
    fmv_at_investment=500_000,
    current_fmv=750_000,
    state="IL",
    is_rural=True,
)

benefits_rural = calculate_benefits(
    rural_inv,
    current_fmv=750_000,
    exit_date="2037-03-15",
)
benefits_rural.summary()

print(f"Standard step-up:  {benefits_std.stepup_pct*100:.0f}%  (${benefits_std.stepup_amount:,.0f})")
print(f"Rural step-up:     {benefits_rural.stepup_pct*100:.0f}%  (${benefits_rural.stepup_amount:,.0f})")
print(f"Rural advantage:   ${benefits_rural.total_tax_benefit - benefits_std.total_tax_benefit:,.0f} additional benefit")


## 7. Full Scenario Comparison

Compare all four strategies for a $1MM capital gain:
1. No OZ investment — pay tax immediately
2. OZ 1.0 standard QOF
3. OZ 2.0 standard QOF
4. OZ 2.0 Rural QORF (enhanced)


In [ ]:
scenarios = compare_scenarios(
    capital_gain=1_000_000,
    investment_date="2027-01-01",
    current_fmv=1_400_000,
    exit_date="2037-01-01",
)

print(f"\nSummary — $1MM Capital Gain, 10-year hold, $1.4MM exit FMV:")
print(f"  No OZ:          Tax = ${scenarios['no_oz_tax']:,.0f}")
print(f"  OZ 1.0:         Benefit = ${scenarios['oz1_benefit']:,.0f}")
print(f"  OZ 2.0:         Benefit = ${scenarios['oz2_benefit']:,.0f}")
print(f"  OZ 2.0 Rural:   Benefit = ${scenarios['oz2_rural_benefit']:,.0f}")


## 8. Holding Period Sensitivity

How do benefits change as the holding period increases?


In [ ]:
holding_periods = [3, 5, 7, 10, 15, 20]

print(f"{'Hold (yrs)':<12} {'Step-Up %':<12} {'Exclusion ($)':<18} {'Total Benefit ($)'}")
print("-" * 60)

for years in holding_periods:
    from datetime import datetime
    from dateutil.relativedelta import relativedelta

    exit_dt = (datetime(2027, 1, 1) + relativedelta(years=years)).strftime("%Y-%m-%d")
    b = calculate_benefits(
        OZInvestment(
            id=f"TEST{years}", investor_name="X", fund_name="X",
            fund_type="qof", oz_version="oz2", tract_id="17031840100",
            investment_type="real_estate", capital_gain_invested=1_000_000,
            investment_date="2027-01-01", fmv_at_investment=1_000_000,
            current_fmv=1_400_000, is_rural=False,
        ),
        current_fmv=1_400_000,
        exit_date=exit_dt,
    )
    print(f"{years:<12} {b.stepup_pct*100:.0f}%{'':<9} "
          f"${b.excluded_appreciation:<17,.0f} ${b.total_tax_benefit:,.0f}")


## 9. Portfolio Tracking

In [ ]:
portfolio = OZPortfolio(name="Family OZ Portfolio 2027")

investments = [
    OZInvestment(
        id="P001", investor_name="Jay Patel",
        fund_name="Chicago South Side OZ Fund",
        fund_type="qof", oz_version="oz2",
        tract_id="17031840100", investment_type="real_estate",
        capital_gain_invested=750_000, investment_date="2027-03-15",
        fmv_at_investment=750_000, current_fmv=900_000,
        state="IL", is_rural=False,
    ),
    OZInvestment(
        id="P002", investor_name="Jay Patel",
        fund_name="Rural Illinois QORF",
        fund_type="qorf", oz_version="oz2",
        tract_id="17019000100", investment_type="real_estate",
        capital_gain_invested=500_000, investment_date="2027-06-01",
        fmv_at_investment=500_000, current_fmv=600_000,
        state="IL", is_rural=True,
    ),
    OZInvestment(
        id="P003", investor_name="Jay Patel",
        fund_name="Detroit OZ Business Fund",
        fund_type="qof", oz_version="oz2",
        tract_id="26163518300", investment_type="operating_business",
        capital_gain_invested=250_000, investment_date="2027-09-01",
        fmv_at_investment=250_000, current_fmv=300_000,
        state="MI", is_rural=False,
    ),
]

for inv in investments:
    portfolio.add(inv)

portfolio.summary()


In [ ]:
df = portfolio.to_dataframe()
print("Portfolio DataFrame:")
print(df[["fund", "fund_type", "oz_version", "capital_gain",
          "is_rural", "state"]].to_string(index=False))


## Summary

This notebook demonstrated the full oz-tracker workflow:

1. **OZ 1.0 eligibility** — check current designated tracts
2. **OZ 2.0 eligibility** — screen for 2027 designations under stricter rules
3. **Tract comparison** — identify which tracts are losing/gaining status
4. **QOF tax benefits** — calculate deferral, step-up, and exclusion
5. **Rural QORF** — model enhanced 30% step-up benefit
6. **Scenario comparison** — compare all four investment strategies
7. **Holding period sensitivity** — optimize exit timing
8. **Portfolio tracking** — aggregate across multiple QOF investments

**Key takeaway for investors:** The 2026 window is critical — investors must
decide whether to deploy under OZ 1.0 before 12/31/2026 or wait for OZ 2.0
enhanced benefits starting 1/1/2027. Rural QORF investments offer the highest
tax benefit with 30% basis step-up vs 10% for standard urban QOFs.

**GitHub:** https://github.com/Jaypatel1511/oz-tracker
**PyPI:** https://pypi.org/project/oz-tracker
